# Continuous-Time Smartphone Battery Drain Model
This notebook implements the complete system of mathematical equations as specified in the **MCM 2026 Project Report**. It utilizes the **2RC-Thevenin Equivalent Circuit Model** combined with physical power laws for hardware components.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# ==========================================
# 1. CONSTANTS & PARAMETERS (Table 1/Appendix D)
# ==========================================
Q_CAP = 3274 * 3600  # Capacity in Coulombs (As) [Eq 1]
V_CUT = 3.0          # Minimum voltage cutoff
V_NOM = 3.7          # Nominal voltage
R0 = 0.0233          # Internal Ohmic Resistance (Ohms)
R1, C1 = 0.015, 2000 # Short-term RC branch parameters
R2, C2 = 0.030, 10000# Long-term RC branch parameters
ETA = 0.98           # Coulombic Efficiency
P_IDLE = 0.05        # Baseline idle power (W)
P_TAIL_5G = 0.5      # Tail state power (W)
TAU_TAIL = 1.5       # Tail state duration (s)
ALPHA_LIGHT = 0.8    # Light mode coefficient
ALPHA_DARK = 0.1     # Dark mode coefficient

## 2. Component Equations
Implementing equations (1) through (15) as derived in the report.

In [ ]:
def get_ocv(soc):
    """[Eq 20/Appendix A] Non-linear Open Circuit Voltage curve"""
    s = np.clip(soc, 0.001, 1.0)
    # Standard Li-Ion discharge curve approximation
    return 3.0 + 1.0 * s + 0.2 * np.log(s + 0.01)

def get_p_cpu(freq, util, cores=8):
    """[Eq 245] CPU Power Model: P = alpha * V^2 * f * U + P_static"""
    # V(f) is the voltage required to sustain frequency f
    voltage = 0.8 + 0.4 * freq
    p_dynamic = cores * (0.12 * (voltage**2) * freq * util)
    p_static = 0.05  # Static leakage component
    return p_dynamic + p_static

def get_p_disp(brightness, apr, is_oled=True):
    """[Eq 260] Display Module Power"""
    if is_oled:
        # P = Beta_max * B * Gamma + P_base
        return 1.85 * brightness * apr + 0.03
    else:
        # P = C_lcd * B + P_driver
        return 1.2 * brightness + 0.1

def get_p_net(t, last_tx, signal_strength, active=True):
    """[Eq 273] Networking (Modem and Tail Energy)"""
    # Delta_signal(L) scaling factor inversely proportional to signal
    delta_sig = 10**((-signal_strength - 80) / 18)
    if active:
        # P = Delta * (D_rate * E_bit + P_tail) + P_idle
        return delta_sig * 2.15 + P_IDLE
    elif t < last_tx + TAU_TAIL:
        # Inactivity timer state (Tail state)
        return P_TAIL_5G
    return P_IDLE

def get_p_misc(activations):
    """[Eq 294] Other Hardware Parasitics (RAM, Cam, GPS)"""
    return 0.1 + 0.2 * activations.get('gps', 0) + 0.1 * activations.get('cam', 0)

## 3. The Governing System [Equations 352]
Combining the hardware drainers into the full state-space derivation.

In [ ]:
def battery_system(y, t, profile):
    """
    y = [SOC (S), V_p1, V_p2]
    Governed by dS/dt and polarization voltage derivatives.
    """
    soc, vp1, vp2 = y
    
    # 1. Total Power Demand (P_sys)
    pcpu = get_p_cpu(profile['f'], profile['u'])
    pdisp = get_p_disp(profile['b'], profile['apr'])
    pnet = get_p_net(t, profile['last_tx'], profile['signal'], profile['net_active'])
    pmisc = get_p_misc(profile['misc'])
    
    p_total = pcpu + pdisp + pnet + pmisc
    
    # Apply Low Power Mode (LPM) limiters [Eq 362]
    if profile.get('lpm', False):
        # LPM reduces CPU freq cap and display brightness
        p_total *= 0.60
        
    # 2. Estimate Current Load (I_load)
    voc = get_ocv(soc)
    # Using terminal voltage derivation: V_t = V_oc - I*R0 - Vp1 - Vp2
    # P = V_t * I -> R0*I^2 - (V_oc - Vp1 - Vp2)*I + P = 0
    v_diff = voc - vp1 - vp2
    a, b, c = R0, -v_diff, p_total
    discriminant = b**2 - 4*a*c
    
    if discriminant < 0:
        # Terminal voltage fell below threshold
        i_load = p_total / 2.5
    else:
        # Real root for discharge current
        i_load = (-b - np.sqrt(discriminant)) / (2*a)
        
    # 3. State Derivatives
    # dS/dt: Normalized SOC change [Eq 192]
    dsoc = - (1 / Q_CAP) * (i_load / ETA)
    
    # dVp1/dt: Polarization transient 1 [Eq 214]
    dvp1 = - (vp1 / (R1 * C1)) + (i_load / C1)
    
    # dVp2/dt: Polarization transient 2 [Eq 215]
    dvp2 = - (vp2 / (R2 * C2)) + (i_load / C2)
    
    return [dsoc, dvp1, dvp2]

## 4. Full Day Simulation (Power vs. Eco Profiles)
This simulates the SOC trajectory under different load conditions over 12 hours.

In [ ]:
t = np.linspace(0, 12 * 3600, 5000) # 12 hours
y0 = [1.0, 0, 0] # Start at 100%

profiles = {
    'Heavy Power User (5G, Gaming, Bright)': {
        'f': 1.0, 'u': 0.98, 'b': 1.0, 'apr': 0.8,
        'signal': -110, 'net_active': True, 'last_tx': 0, 
        'misc': {'gps': 1, 'cam': 0}, 'lpm': False
    },
    'Standard User': {
        'f': 0.6, 'u': 0.4, 'b': 0.5, 'apr': 0.4,
        'signal': -90, 'net_active': False, 'last_tx': 100, 
        'misc': {'gps': 0, 'cam': 0}, 'lpm': False
    },
    'Limited User (LPM, Dark Mode)': {
        'f': 0.4, 'u': 0.2, 'b': 0.3, 'apr': 0.1,
        'signal': -80, 'net_active': False, 'last_tx': -100, 
        'misc': {'gps': 0, 'cam': 0}, 'lpm': True
    }
}

plt.figure(figsize=(14, 8))
for name, prof in profiles.items():
    sol = odeint(battery_system, y0, t, args=(prof,))
    plt.plot(t/3600, sol[:, 0]*100, label=name, lw=2.5)

plt.axhline(0, color='black', alpha=0.5, ls='--')
plt.title("Smartphone State of Charge (SOC) Prediction [Continuous-Time Model]", fontsize=16)
plt.xlabel("Time (Hours)", fontsize=12)
plt.ylabel("Battery Percentage (%)", fontsize=12)
plt.legend(fontsize=10)
plt.grid(alpha=0.2)
plt.ylim(-5, 105)
plt.tight_layout()
plt.show()